# Shohoj Lipi - Sentence Router Training (Kaggle/Colab)
This notebook trains the BanglaBERT model on the 96K BengaliReadability sentence corpus. Once trained, it exports the model to ONNX for fast CPU inference on our FastAPI backend.

In [1]:
!pip install transformers datasets evaluate optimum[onnxruntime] scikit-learn accelerate


INFO: pip is looking at multiple versions of optimum-onnx[onnxruntime] to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 87.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 12.1 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bi

In [2]:
import urllib.request
import urllib.parse
import os
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate
import numpy as np

# 1. Download datasets directly from GitHub API
os.makedirs("data", exist_ok=True)
base_url = "https://raw.githubusercontent.com/tafseer-nayeem/BengaliReadability/main/Data/Learning/"
for f in ["train.csv", "valid.csv", "test.csv"]:
    print(f"Downloading {f}...")
    urllib.request.urlretrieve(base_url + f, f"data/{f}")
print("Datasets downloaded!")

2026-06-30 17:07:12.749011: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782839233.179757      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782839233.290156      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782839234.287618      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782839234.287675      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782839234.287678      58 computation_placer.cc:177] computation placer alr

Datasets downloaded!


In [3]:
# 2. Load data
df_train = pd.read_csv("data/train.csv")
df_valid = pd.read_csv("data/valid.csv")

# Parse columns
text_col = df_train.columns[0]
label_col = df_train.columns[1]

if df_train[label_col].dtype == object:
    labels = sorted(df_train[label_col].unique())
    label_map = {l: i for i, l in enumerate(labels)}
    df_train[label_col] = df_train[label_col].map(label_map)
    df_valid[label_col] = df_valid[label_col].map(label_map)
else:
    label_map = {0: "Simple", 1: "Complex"}

ds = DatasetDict({
    "train": Dataset.from_pandas(df_train[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"})),
    "validation": Dataset.from_pandas(df_valid[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"}))
})

In [4]:
# 3. Preprocess with BanglaBERT tokenizer
MODEL_NAME = "csebuetnlp/banglabert_small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_ds = ds.map(preprocess, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/91935 [00:00<?, ? examples/s]

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

In [ ]:
# 4. Define and Train the Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=len(label_map),
    id2label={v: k for k, v in label_map.items()} if isinstance(list(label_map.keys())[0], str) else label_map,
    label2id={k: v for k, v in label_map.items()} if isinstance(list(label_map.keys())[0], str) else {v: k for k, v in label_map.items()}
)

def compute_metrics(eval_pred):
    metric = evaluate.load("accuracy")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./sentence_router_pt",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=False,
    bf16=False,
    report_to="none", # Will use Kaggle GPU natively
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("sentence_router_pt")

pytorch_model.bin:   0%|          | 0.00/55.0M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert_small and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_58/2488360297.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


model.safetensors:   0%|          | 0.00/55.0M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


In [ ]:
# 5. Export to ONNX
!optimum-cli export onnx --model sentence_router_pt --task text-classification sentence_router_onnx
!optimum-cli onnxruntime quantize --onnx_model sentence_router_onnx --avx512 -o sentence_router_onnx/quantized

print("✅ Exported and quantized to ONNX!")
print("Download the 'sentence_router_onnx/quantized' folder and place it in 'models/' in your backend.")